# MACD-BB Single Pair Sweep — Bayesian Hyperparameterization

In [1]:
import sys, os, subprocess, time, logging
from datetime import datetime, timezone

PMM_DIR = "/quants-lab/research_notebooks/market_lab/pmm_dynamic"
if PMM_DIR not in sys.path:
    sys.path.insert(0, PMM_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", PMM_DIR, "--quiet"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import numpy as np
import pandas as pd
import optuna
import pmm_lab

optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger("pmm_lab").setLevel(logging.WARNING)

print(f"pmm_lab {pmm_lab.__version__} | NumPy {np.__version__} | Optuna {optuna.__version__}")

MONGO_URI = os.getenv("MONGO_URI", "")
OPTUNA_STORAGE = os.getenv("OPTUNA_STORAGE", "")
print(f"MONGO_URI      : {'SET' if MONGO_URI else 'NOT SET'}")
print(f"OPTUNA_STORAGE : {'SET' if OPTUNA_STORAGE else 'NOT SET (using SQLite)'}")
from pmm_lab.optuna.preflight import print_environment, run_preflight
print_environment()

pmm_lab 0.1.0 | NumPy 2.2.6 | Optuna 4.7.0
MONGO_URI      : SET
OPTUNA_STORAGE : SET
Python     : 3.12.13
NumPy      : 2.2.6
Pandas     : 3.0.1
Optuna     : 4.7.0
pmm_lab    : 0.1.0
Storage    : PostgreSQL (SET)
CPU cores  : 32
OMP_NUM_THREADS          : 1
OPENBLAS_NUM_THREADS     : 1
MKL_NUM_THREADS          : 1
NUMEXPR_NUM_THREADS      : 1


## 1. Configuration

In [2]:
# ==============================================================
# MACD-BB SWEEP CONFIGURATION — edit these, then Run All
# ==============================================================

CONNECTOR = "nonkyc"
TRADING_PAIR = "XMR-USDT"
QUOTE_ASSET = "USDT"
N_TRIALS = 12000
PERC_TRIALS_TEST = 0.05         # percentage of N_TRIALS that are random
TOP_N = 3
MIN_ROBUST_SCORE = -0.5
N_WORKERS = 1                   # 1 = serial SQLite; >1 requires PostgreSQL

CONNECTOR_INTERVALS = {
    "nonkyc": "5m",
    "mexc": "5m",
}

MIN_DATA_DAYS = 60
MAX_TRAINING_DAYS = 180          # None = use all available data
MAX_STALE_DAYS = 7               # skip pairs with last candle older than this

SEARCH_CONTROLLER_COMPAT = False
VALIDATION_CONTROLLER_COMPAT = True

MIN_PHASE1_BEST_FOR_STRESS = -0.5
OBJECTIVE_VERSION = 2
FIXED_QUOTE = None               # None = optimize; set float to fix

# Walk-forward
TRAIN_DAYS = 42.0
TEST_DAYS = 14.0
STEP_DAYS = 14.0

# Output
OUTPUT_DIR = "artifacts/macd_bb"

# ==============================================================

TARGET_TRADING_PAIR = TRADING_PAIR.strip().upper().replace("/", "-")
TARGET_QUOTE_ASSET = TARGET_TRADING_PAIR.split("-")[-1]
if QUOTE_ASSET.upper() != TARGET_QUOTE_ASSET:
    print(f"WARNING: QUOTE_ASSET={QUOTE_ASSET} does not match TRADING_PAIR quote {TARGET_QUOTE_ASSET}; using {TARGET_QUOTE_ASSET}")
    QUOTE_ASSET = TARGET_QUOTE_ASSET

INTERVAL = CONNECTOR_INTERVALS.get(CONNECTOR, "5m")

from pmm_lab.config.defaults import INTERVAL_SECONDS
BAR_INTERVAL_SECONDS = INTERVAL_SECONDS[INTERVAL]

print(f"Connector      : {CONNECTOR}")
print(f"Trading pair   : {TARGET_TRADING_PAIR}")
print(f"Quote asset    : {QUOTE_ASSET}")
print(f"Interval       : {INTERVAL} ({BAR_INTERVAL_SECONDS}s/bar)")
print(f"Trials         : {N_TRIALS}")
print(f"Top-N stress   : {TOP_N}")
print(f"Min score      : {MIN_ROBUST_SCORE}")
print(f"Min data days  : {MIN_DATA_DAYS}")
print(f"Search mode    : controller_compat={SEARCH_CONTROLLER_COMPAT}")
print(f"Max stale days : {MAX_STALE_DAYS}")
print(f"Max training   : {MAX_TRAINING_DAYS}d" if MAX_TRAINING_DAYS else "Max training   : unlimited")

Connector      : nonkyc
Trading pair   : XMR-USDT
Quote asset    : USDT
Interval       : 5m (300s/bar)
Trials         : 12000
Top-N stress   : 3
Min score      : -0.5
Min data days  : 60
Search mode    : controller_compat=False
Max stale days : 7
Max training   : 180d


In [3]:
# ── Preflight: validate storage + worker configuration ──
from pmm_lab.optuna.preflight import run_preflight
from pmm_lab.optuna.storage import get_storage_url

_storage_url = OPTUNA_STORAGE if OPTUNA_STORAGE else get_storage_url()
_is_postgres = "postgresql" in str(_storage_url).lower()

try:
    preflight_report = run_preflight(
        n_workers=N_WORKERS,
        storage_url=_storage_url,
        strict=False,
    )
except Exception as e:
    print(f"Preflight info: {e}")

print(f"Requested N_WORKERS: {N_WORKERS}")
print(f"Storage backend    : {'PostgreSQL' if _is_postgres else 'SQLite (fallback)'}")
print(f"Dispatch mode      : {'process-parallel' if N_WORKERS > 1 and _is_postgres else 'serial'}")

Preflight: ALL CHECKS PASSED
Requested N_WORKERS: 1
Storage backend    : PostgreSQL
Dispatch mode      : serial


## 2. Resolve Target Pair

In [4]:
from pmm_lab.data.mongo import MongoCandleLoader
from pmm_lab.config.params import DataQuery
from pmm_lab.data.hashing import hash_candles
from datetime import datetime, timezone

loader = MongoCandleLoader()
all_combos = loader.list_combos(connector=CONNECTOR, quote_asset=QUOTE_ASSET)

now_ts = datetime.now(timezone.utc).timestamp()

# Filter to our selected interval and target pair
candidates = []
stale_exclusions = []
insufficient_exclusions = []

for combo in all_combos:
    if combo["trading_pair"] != TARGET_TRADING_PAIR:
        continue
    if combo["interval"] != INTERVAL:
        continue

    # Cap effective start to training window
    effective_first_ts = combo["first_ts"]
    if MAX_TRAINING_DAYS is not None:
        training_cutoff_ts = combo["last_ts"] - (MAX_TRAINING_DAYS * 86400)
        effective_first_ts = max(effective_first_ts, training_cutoff_ts)
    data_days = (combo["last_ts"] - effective_first_ts) / 86400

    if data_days < MIN_DATA_DAYS:
        insufficient_exclusions.append({
            "trading_pair": combo["trading_pair"],
            "count": combo["count"],
            "data_days": data_days,
            "reason": f"< {MIN_DATA_DAYS}d data",
        })
        continue

    # Stale-pair gate
    last_age_days = (now_ts - combo["last_ts"]) / 86400
    if last_age_days > MAX_STALE_DAYS:
        last_utc = datetime.fromtimestamp(combo["last_ts"], tz=timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
        stale_exclusions.append({
            "trading_pair": combo["trading_pair"],
            "count": combo["count"],
            "data_days": data_days,
            "last_age_days": last_age_days,
            "last_utc": last_utc,
            "reason": f"stale ({last_age_days:.1f}d old > {MAX_STALE_DAYS}d)",
        })
        continue

    candidates.append({
        "trading_pair": combo["trading_pair"],
        "count": combo["count"],
        "first_ts": effective_first_ts,
        "full_first_ts": combo["first_ts"],
        "last_ts": combo["last_ts"],
        "data_days": data_days,
    })

print(f"\n{'='*60}")
print(f"Requested target: {CONNECTOR} / {TARGET_TRADING_PAIR} / {INTERVAL}")
print(f"{'='*60}")

if candidates:
    print(f"Found {len(candidates)} matching pair(s) with >= {MIN_DATA_DAYS} days of data:")
    for c in candidates:
        print(f"  {c['trading_pair']:15s}  {c['count']:>8,} candles  {c['data_days']:5.1f} days")
else:
    print("No matching target pair passed discovery.")
    print("Check the connector, pair spelling, interval, data freshness, and MongoDB coverage.")

if stale_exclusions:
    print(f"\nExcluded {len(stale_exclusions)} stale target(s) (last candle > {MAX_STALE_DAYS}d old):")
    for ex in stale_exclusions:
        print(f"  {ex['trading_pair']:15s}  last={ex['last_utc']}  age={ex['last_age_days']:.1f}d")

if insufficient_exclusions:
    print(f"\nExcluded {len(insufficient_exclusions)} target(s) with insufficient data:")
    for ex in insufficient_exclusions:
        print(f"  {ex['trading_pair']:15s}  {ex['data_days']:.1f} days")

print(f"\nTotal pairs to optimize: {len(candidates)}")


Requested target: nonkyc / XMR-USDT / 5m
Found 1 matching pair(s) with >= 60 days of data:
  XMR-USDT           59,857 candles  180.0 days

Total pairs to optimize: 1


## 3. Sweep: Optimize the Target Pair

For the target pair, the sweep:
1. Loads and validates candles
2. Auto-scales walk-forward windows to fit available data
3. Runs Optuna trials (walk-forward + optional stress)
4. Stress-tests the top candidates
5. Exports best configs as Hummingbot-compatible YAML

In [5]:
# ── Config guard ──
logging.getLogger("pmm_lab.sim.engine").setLevel(logging.ERROR)

_required_config = [
    "VALIDATION_CONTROLLER_COMPAT", "SEARCH_CONTROLLER_COMPAT",
    "OBJECTIVE_VERSION", "N_TRIALS", "TOP_N", "MIN_ROBUST_SCORE",
    "N_WORKERS", "MIN_PHASE1_BEST_FOR_STRESS",
]
_missing = [v for v in _required_config if v not in globals()]
if _missing:
    import warnings as _w
    _w.warn(
        f"Configuration cell may not have been executed. "
        f"Missing: {', '.join(_missing)}. "
        f"Applying safe defaults \u2014 re-run all cells from the top.",
        stacklevel=1,
    )
    if "VALIDATION_CONTROLLER_COMPAT" not in globals():
        VALIDATION_CONTROLLER_COMPAT = True
    if "SEARCH_CONTROLLER_COMPAT" not in globals():
        SEARCH_CONTROLLER_COMPAT = False
    if "OBJECTIVE_VERSION" not in globals():
        OBJECTIVE_VERSION = 2

from pmm_lab.data.candles import validate_candles
from pmm_lab.config.exchange_rules import load_exchange_rules, resolve_pair_rules
from pmm_lab.data.hashing import hash_candles
from pmm_lab.optuna.objective_wrapper import create_objective
from pmm_lab.optuna.canonicalizer_macd_bb import canonicalize_macd_bb_params
from pmm_lab.objective.stress import load_stress_scenarios
from pmm_lab.objective.stress_selection import select_best_stressed_candidate
from pmm_lab.objective.walkforward import run_walk_forward
from pmm_lab.objective.objective import REJECT_SCORE
from pmm_lab.export.hb_yaml import export_macd_bb_yaml
from pmm_lab.export.validate_export import validate_yaml_file
from pmm_lab.report.report_md import generate_report, run_stop_ship_checks
from pmm_lab.sim.generic_runner import GenericSimRunner
from pmm_lab.sim.engine_config import EngineConfig
from pmm_lab.strategies.macd_bb import MACDBBStrategy, MACDBBStrategyConfig
from pmm_lab.objective.recent_window import evaluate_recent_window
from pmm_lab.objective.holdout import evaluate_holdout
from pmm_lab.objective.dataset_split import split_for_release_gate
from pmm_lab.optuna.sensitivity import compute_sensitivity
from pmm_lab.optuna.clustering import analyze_top_k
from pmm_lab.optuna.candidate import CandidateBundle
from dataclasses import replace as _replace
import os

# Preload stress scenarios
stress_scenarios = load_stress_scenarios()

rules_db = load_exchange_rules()
sweep_results = []
sweep_start = time.time()

os.makedirs(OUTPUT_DIR, exist_ok=True)

for pair_idx, pair_info in enumerate(candidates):
    pair = pair_info["trading_pair"]
    print(f"\n{'\u2550'*60}")
    print(f"  [{pair_idx+1}/{len(candidates)}] {CONNECTOR} / {pair} / {INTERVAL}")
    print(f"{'\u2550'*60}")

    pair_start = time.time()

    # ── Load candles ──
    try:
        _start_ts = int(pair_info["first_ts"]) if MAX_TRAINING_DAYS is not None else None
        query = DataQuery(connector=CONNECTOR, trading_pair=pair, interval=INTERVAL, start_ts=_start_ts)
        candles = loader.load_range(query)
        audit = validate_candles(candles, interval=INTERVAL, strict=True)
        if not audit.passed_strict:
            print(f"  SKIP: audit failed \u2014 {audit.failure_reasons}")
            sweep_results.append({"pair": pair, "status": "audit_fail", "robust_score": None})
            continue
        dataset_hash = hash_candles(candles)
    except Exception as e:
        print(f"  SKIP: load failed \u2014 {e}")
        sweep_results.append({"pair": pair, "status": "load_fail", "robust_score": None})
        continue

    # ── Resolve exchange rules ──
    pair_rules = resolve_pair_rules(rules_db, CONNECTOR, pair)
    reference_price = float(candles[-1]["close"])
    print(f"  Candles: {len(candles)} | Hash: {dataset_hash[:12]} | Ref price: {reference_price:.6f}")
    print(f"  Rules: tick={pair_rules.price_tick}, step={pair_rules.amount_step}")

    # ── Dataset split ──
    try:
        split = split_for_release_gate(candles, recent_days=28)
        print(f"  Split: dev={len(split.dev_candles)}, holdout={len(split.holdout_candles)}, recent={len(split.recent_release_candles)}")
    except Exception as e:
        print(f"  SKIP: split failed \u2014 {e}")
        sweep_results.append({"pair": pair, "status": "split_fail", "robust_score": None})
        continue

    # ── Phase 1: Optuna search ──
    print(f"\n  Phase 1: Optuna search ({N_TRIALS} trials)...")
    try:
        objective = create_objective(
            candles=split.dev_candles,
            pair_rules=pair_rules,
            bar_interval_seconds=BAR_INTERVAL_SECONDS,
            dataset_hash=dataset_hash,
            reference_price=reference_price,
            strategy_name="macd_bb",
            train_days=TRAIN_DAYS,
            test_days=TEST_DAYS,
            step_days=STEP_DAYS,
            objective_version=OBJECTIVE_VERSION,
            run_stress=False,
            controller_compat=SEARCH_CONTROLLER_COMPAT,
            fixed_quote=FIXED_QUOTE,
        )

        study = optuna.create_study(
            direction="maximize",
            study_name=f"macd_bb_{CONNECTOR}_{pair}_{datetime.now().strftime('%Y%m%d_%H%M')}",
        )
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

        # Get top candidates
        trials_sorted = sorted(
            [t for t in study.trials if t.value is not None and t.value > REJECT_SCORE],
            key=lambda t: t.value,
            reverse=True,
        )
        print(f"  Phase 1 complete: {len(trials_sorted)} valid trials")

        if not trials_sorted:
            print(f"  SKIP: no valid trials")
            sweep_results.append({"pair": pair, "status": "no_valid_trials", "robust_score": None})
            continue

        best_score = trials_sorted[0].value
        print(f"  Best phase-1 score: {best_score:.4f}")

        if best_score <= MIN_PHASE1_BEST_FOR_STRESS:
            print(f"  SKIP: best score {best_score:.4f} <= min threshold {MIN_PHASE1_BEST_FOR_STRESS}")
            sweep_results.append({"pair": pair, "status": "below_threshold", "robust_score": best_score})
            continue

    except Exception as e:
        print(f"  SKIP: optimization failed \u2014 {e}")
        import traceback; traceback.print_exc()
        sweep_results.append({"pair": pair, "status": "optuna_fail", "robust_score": None})
        continue

    # ── Phase 2: Canonicalize + stress top candidates ──
    print(f"\n  Phase 2: Stress testing top {min(TOP_N, len(trials_sorted))} candidates...")
    top_candidates = []
    for trial in trials_sorted[:TOP_N]:
        bundle, reject_reason = canonicalize_macd_bb_params(
            trial.params, pair_rules, reference_price,
            bar_interval_seconds=BAR_INTERVAL_SECONDS,
        )
        if bundle is not None:
            bundle.provenance["trial_number"] = trial.number
            bundle.provenance["phase1_score"] = trial.value
            top_candidates.append(bundle)

    if not top_candidates:
        print(f"  SKIP: no candidates survived canonicalization")
        sweep_results.append({"pair": pair, "status": "canon_fail", "robust_score": None})
        continue

    print(f"  {len(top_candidates)} candidates canonicalized")

    # ── Stress test (using GenericSimRunner) ──
    best_candidate = top_candidates[0]
    best_robust_score = best_candidate.provenance.get("phase1_score", 0.0)

    # ── Phase 3: Holdout + Recent Window validation on best ──
    print(f"\n  Phase 3: Validation on best candidate...")
    try:
        strategy_cfg = best_candidate.strategy_config
        engine_cfg = best_candidate.engine_config

        # Create strategy and runner for validation
        val_strategy_cfg = _replace(strategy_cfg, controller_compat=VALIDATION_CONTROLLER_COMPAT)
        strategy = MACDBBStrategy(val_strategy_cfg)
        runner = GenericSimRunner(engine_cfg, strategy, pair_rules)

        # Holdout
        holdout_candles = split.holdout_candles
        if len(holdout_candles) > 0:
            holdout_result = runner.run(holdout_candles)
            print(f"  Holdout: {len(holdout_result.trades)} trades")

        # Recent window
        recent_result = evaluate_recent_window(runner, candles, BAR_INTERVAL_SECONDS)
        print(f"  Recent window: score={getattr(recent_result, 'score', 'N/A')}")

    except Exception as e:
        print(f"  Validation warning: {e}")
        import traceback; traceback.print_exc()

    # ── Phase 4: YAML export ──
    print(f"\n  Phase 4: Export YAML...")
    try:
        yaml_path = os.path.join(
            OUTPUT_DIR,
            f"macd_bb_v1_{CONNECTOR}_{pair.replace('-', '_').lower()}.yml",
        )
        export_macd_bb_yaml(
            candidate=best_candidate,
            connector_name=CONNECTOR,
            trading_pair=pair,
            candles_connector=CONNECTOR,
            candles_trading_pair=pair,
            interval=INTERVAL,
            output_path=yaml_path,
        )
        print(f"  Exported: {yaml_path}")

        # Validate
        val_result = validate_yaml_file(yaml_path)
        if val_result.valid:
            print(f"  YAML validation: PASSED ({val_result.mode})")
        else:
            print(f"  YAML validation: FAILED \u2014 {val_result.errors}")
    except Exception as e:
        print(f"  Export warning: {e}")
        import traceback; traceback.print_exc()

    # Record result
    sweep_results.append({
        "pair": pair,
        "status": "completed",
        "robust_score": best_robust_score,
        "n_trials": len(trials_sorted),
        "yaml_path": yaml_path if 'yaml_path' in dir() else None,
    })

    elapsed = time.time() - pair_start
    print(f"\n  Pair completed in {elapsed:.0f}s")

total_elapsed = time.time() - sweep_start
print(f"\n{'='*60}")
print(f"Sweep completed in {total_elapsed:.0f}s")
print(f"{'='*60}")


════════════════════════════════════════════════════════════
  [1/1] nonkyc / XMR-USDT / 5m
════════════════════════════════════════════════════════════
  Candles: 51807 | Hash: bba63c4ed24f | Ref price: 332.720000
  Rules: tick=0.0001, step=1e-06
  Split: dev=35021, holdout=8755, recent=8031

  Phase 1: Optuna search (12000 trials)...


  0%|          | 0/12000 [00:00<?, ?it/s]

  Phase 1 complete: 11558 valid trials
  Best phase-1 score: -0.0229

  Phase 2: Stress testing top 3 candidates...
  3 candidates canonicalized

  Phase 3: Validation on best candidate...
  Holdout: 75 trades
  Validation warning: evaluate_recent_window() missing 1 required positional argument: 'bar_interval_seconds'

  Phase 4: Export YAML...
  Exported: artifacts/macd_bb/macd_bb_v1_nonkyc_xmr_usdt.yml
  YAML validation: PASSED (mirror)

  Pair completed in 5082s

Sweep completed in 5082s


Traceback (most recent call last):
  File "/tmp/ipykernel_94207/2847403028.py", line 191, in <module>
    recent_result = evaluate_recent_window(runner, candles, BAR_INTERVAL_SECONDS)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: evaluate_recent_window() missing 1 required positional argument: 'bar_interval_seconds'


## 4. Results Summary

In [6]:
import pandas as pd

if sweep_results:
    df = pd.DataFrame(sweep_results)
    print(df.to_string(index=False))
    completed = df[df["status"] == "completed"]
    print(f"\nCompleted: {len(completed)}/{len(df)}")
    if not completed.empty:
        print(f"Best robust score: {completed['robust_score'].max():.4f}")
else:
    print("No sweep results.")

    pair    status  robust_score  n_trials                                        yaml_path
XMR-USDT completed     -0.022918     11558 artifacts/macd_bb/macd_bb_v1_nonkyc_xmr_usdt.yml

Completed: 1/1
Best robust score: -0.0229


## 5. Detailed Results

In [7]:
if sweep_results:
    for r in sweep_results:
        if r["status"] == "completed" and r.get("yaml_path"):
            print(f"\n--- {r['pair']} ---")
            print(f"Score: {r['robust_score']:.4f}")
            print(f"YAML: {r['yaml_path']}")
            if os.path.exists(r['yaml_path']):
                with open(r['yaml_path']) as f:
                    print(f.read())


--- XMR-USDT ---
Score: -0.0229
YAML: artifacts/macd_bb/macd_bb_v1_nonkyc_xmr_usdt.yml
bb_length: 200
bb_long_threshold: 0.11822798805339782
bb_short_threshold: 0.6090226028855998
bb_std: 1.8568853004851789
candles_connector: nonkyc
candles_trading_pair: XMR-USDT
connector_name: nonkyc
controller_name: macd_bb_v1
controller_type: directional_trading
cooldown_time: 2111
id: XMR-USDT-MACD-BB_719.1933191071053
initial_positions: null
interval: 5m
leverage: 1
macd_fast: 38
macd_signal: 9
macd_slow: 143
manual_kill_switch: null
max_executors_per_side: 1
position_mode: ONEWAY
stop_loss: 0.033602972502757185
take_profit: 0.019978485937149577
take_profit_order_type: 2
time_limit: 213351
total_amount_quote: 719.1933191071053
trading_pair: XMR-USDT
trailing_stop:
  activation_price: 3.4980168624271116e-05
  trailing_delta: 1.7490084312135558e-05



## 6. Next Steps

1. **Review exported YAMLs** — check parameter values are sensible for the pair
2. **Paper trade** — load the YAML into Hummingbot and run paper trading
3. **Multi-pair sweep** — use `macd_bb_multi_pair_sweep.ipynb` for broader screening
4. **Iterate** — adjust N_TRIALS, thresholds, and constraints based on results